Ноутбук отвечает за дообучение PromptRetriever с LoRA-адаптерами и квантованием. В нем оставлен код обучения, который использовался для финального эксперимента.


In [ ]:
!nvidia-smi

!pip uninstall -y bitsandbytes peft -q
!pip install -q peft==0.13.2 bitsandbytes==0.46.1 sentencepiece protobuf accelerate datasets

import os
import json
import random
import gc
import torch
import torch.nn.functional as F
import wandb

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

import bitsandbytes as bnb


In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()


HF_TOKEN = os.environ.get("HF_TOKEN", "")
login(token=HF_TOKEN) if HF_TOKEN else None

TRAIN_FILE = "/kaggle/input/datasets/sukiss/prmptr/tevatron_ru_promptriever_train (2).jsonl"

BASE_MODEL = "meta-llama/Llama-2-7b-hf"
PROMPTRIEVER_ADAPTER = "samaya-ai/promptriever-llama2-7b-v1"

OUTPUT_DIR = "/kaggle/working/promptriever_llama2_7b_ru_qlora_t4"
ARCHIVE_PATH = "/kaggle/working/promptriever_llama2_7b_ru_qlora_t4.tar.gz"
LOG_PATH = "/kaggle/working/train_log.jsonl"

BATCH_SIZE = 1
TRAIN_GROUP_SIZE = 6
GRAD_ACCUM = 16

QUERY_MAX_LEN = 192
PASSAGE_MAX_LEN = 128
TEMPERATURE = 0.01

LR = 5e-5
EPOCHS = 1
SAVE_EVERY = 100
SEED = 42

USE_ONLY_CLEAN_ROWS = False

random.seed(SEED)
torch.manual_seed(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)

wandb.init(
    project="ru-promptriever",
    name="promptriever-llama2-7b-qlora-t4-g6",
    config={
        "base_model": BASE_MODEL,
        "promptriever_adapter": PROMPTRIEVER_ADAPTER,
        "train_file": TRAIN_FILE,
        "output_dir": OUTPUT_DIR,
        "batch_size": BATCH_SIZE,
        "train_group_size": TRAIN_GROUP_SIZE,
        "grad_accum": GRAD_ACCUM,
        "query_max_len": QUERY_MAX_LEN,
        "passage_max_len": PASSAGE_MAX_LEN,
        "temperature": TEMPERATURE,
        "learning_rate": LR,
        "epochs": EPOCHS,
        "use_only_clean_rows": USE_ONLY_CLEAN_ROWS,
        "seed": SEED,
    }
)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    PROMPTRIEVER_ADAPTER,
    use_fast=False,
    token=HF_TOKEN if HF_TOKEN else None,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

base = AutoModel.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    token=HF_TOKEN if HF_TOKEN else None,
)

base.config.use_cache = False
base.gradient_checkpointing_enable()
base = prepare_model_for_kbit_training(base)

from huggingface_hub import snapshot_download

ADAPTER_LOCAL = "/kaggle/working/promptriever_adapter_local"

snapshot_download(
    repo_id=PROMPTRIEVER_ADAPTER,
    local_dir=ADAPTER_LOCAL,
    local_dir_use_symlinks=False,
    token=HF_TOKEN if HF_TOKEN else None,
)

print("Adapter files:")
!find /kaggle/working/promptriever_adapter_local -maxdepth 2 -type f | sed -n '1,80p'

adapter_config_path = os.path.join(ADAPTER_LOCAL, "adapter_config.json")
if not os.path.exists(adapter_config_path):
    raise FileNotFoundError(f"adapter_config.json not found at {adapter_config_path}")

model = PeftModel.from_pretrained(
    base,
    ADAPTER_LOCAL,
    is_trainable=True,
)

model.train()
model.print_trainable_parameters()

DEVICE = "cuda"


class PromptrieverInstructionDataset(Dataset):
    def __init__(self, path, use_only_clean_rows=False):
        self.items = []

        raw_rows = 0
        kept = 0
        skipped = {
            "not_instruction": 0,
            "validation_errors": 0,
            "missing_positive": 0,
            "not_enough_negatives": 0,
            "not_enough_original": 0,
            "not_enough_generated": 0,
            "empty_text": 0,
        }

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                raw_rows += 1
                item = json.loads(line)

                meta = item.get("metadata", {})

                if meta.get("has_instruction") is not True:
                    skipped["not_instruction"] += 1
                    continue

                if use_only_clean_rows and item.get("validation_errors"):
                    skipped["validation_errors"] += 1
                    continue

                positives = item.get("positive_passages", [])
                negatives = item.get("negative_passages", [])

                if len(positives) < 1:
                    skipped["missing_positive"] += 1
                    continue

                if len(negatives) < 5:
                    skipped["not_enough_negatives"] += 1
                    continue

                query = item.get("query", "")
                positive = positives[0].get("text", "")

                if not query or not positive:
                    skipped["empty_text"] += 1
                    continue

                original_negatives = []
                generated_negatives = []

                for neg in negatives:
                    docid = str(neg.get("docid", ""))
                    text = neg.get("text", "")

                    if not text:
                        continue

                    if "_gen_" in docid:
                        generated_negatives.append(text)
                    else:
                        original_negatives.append(text)

                if len(original_negatives) < 2:
                    skipped["not_enough_original"] += 1
                    continue

                if len(generated_negatives) < 3:
                    skipped["not_enough_generated"] += 1
                    continue

                self.items.append({
                    "query": query,
                    "positive": positive,
                    "original_negatives": original_negatives,
                    "generated_negatives": generated_negatives,
                    "query_id": item.get("query_id"),
                    "instruction_style": meta.get("instruction_style"),
                })

                kept += 1

        self.raw_rows = raw_rows
        self.kept = kept
        self.skipped = skipped

        print("Raw rows:", raw_rows)
        print("Kept instruction rows:", kept)
        print("Skipped:", skipped)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        original = random.sample(item["original_negatives"], 2)
        generated = random.sample(item["generated_negatives"], 3)

        passages = [item["positive"]] + original + generated

        return {
            "query": item["query"],
            "passages": passages,
        }


def collate_fn(batch):
    queries = [x["query"] for x in batch]

    passages = []
    for x in batch:
        passages.extend(x["passages"])

    return queries, passages


dataset = PromptrieverInstructionDataset(
    TRAIN_FILE,
    use_only_clean_rows=USE_ONLY_CLEAN_ROWS,
)

print("Train rows:", len(dataset))

if len(dataset) == 0:
    raise ValueError("No train rows found. Check TRAIN_FILE and metadata.has_instruction.")

print("Example query:", dataset[0]["query"][:500])
print("Passages per query:", len(dataset[0]["passages"]))

wandb.config.update({
    "raw_rows": dataset.raw_rows,
    "kept_instruction_rows": dataset.kept,
    "skipped_rows": dataset.skipped,
}, allow_val_change=True)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)


def add_eos(texts):
    return [t + tokenizer.eos_token for t in texts]


def eos_pool(last_hidden_state, attention_mask):
    lengths = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
    return last_hidden_state[batch_idx, lengths]


def encode_texts(texts, max_len):
    texts = add_eos(texts)

    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(DEVICE)

    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
    )

    emb = eos_pool(out.last_hidden_state, batch["attention_mask"])
    emb = F.normalize(emb, p=2, dim=-1)

    return emb


optimizer = bnb.optim.PagedAdamW8bit(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
)


with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write(json.dumps({
        "event": "start",
        "train_rows": len(dataset),
        "batch_size": BATCH_SIZE,
        "train_group_size": TRAIN_GROUP_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
    }, ensure_ascii=False) + "\n")


global_step = 0
running_loss = 0.0
examples_seen = 0
passages_seen = 0

model.train()
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    pbar = tqdm(loader, desc=f"epoch {epoch + 1}")

    for step, (queries, passages) in enumerate(pbar):
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            q_emb = encode_texts(queries, QUERY_MAX_LEN)
            p_emb = encode_texts(passages, PASSAGE_MAX_LEN)

            scores = torch.matmul(q_emb, p_emb.T) / TEMPERATURE

            labels = torch.arange(
                q_emb.size(0),
                device=DEVICE,
                dtype=torch.long,
            ) * TRAIN_GROUP_SIZE

            loss = F.cross_entropy(scores, labels)
            loss = loss / GRAD_ACCUM

        loss.backward()
        running_loss += loss.item() * GRAD_ACCUM

        examples_seen += len(queries)
        passages_seen += len(passages)

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                1.0
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            avg_loss = running_loss / GRAD_ACCUM
            running_loss = 0.0

            mem_alloc = torch.cuda.memory_allocated() / 1024**3
            mem_reserved = torch.cuda.memory_reserved() / 1024**3

            log_row = {
                "train/loss": avg_loss,
                "train/global_step": global_step,
                "train/item_step": step + 1,
                "train/epoch": epoch + 1,
                "train/lr": LR,
                "train/examples_seen": examples_seen,
                "train/passages_seen": passages_seen,
                "gpu/memory_allocated_gb": mem_alloc,
                "gpu/memory_reserved_gb": mem_reserved,
            }

            wandb.log(log_row, step=global_step)

            with open(LOG_PATH, "a", encoding="utf-8") as f:
                f.write(json.dumps(log_row, ensure_ascii=False) + "\n")

            pbar.set_postfix({
                "loss": round(avg_loss, 4),
                "global_step": global_step,
                "bs": BATCH_SIZE,
                "group": TRAIN_GROUP_SIZE,
            })

            if global_step % SAVE_EVERY == 0:
                ckpt_dir = f"{OUTPUT_DIR}/checkpoint-{global_step}"
                os.makedirs(ckpt_dir, exist_ok=True)
                model.save_pretrained(ckpt_dir)
                tokenizer.save_pretrained(ckpt_dir)
                print("Saved checkpoint:", ckpt_dir)

if (step + 1) % GRAD_ACCUM != 0:
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad],
        1.0
    )

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    global_step += 1


os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved final adapter:", OUTPUT_DIR)

!tar -czf /kaggle/working/promptriever_llama2_7b_ru_qlora_t4.tar.gz -C /kaggle/working promptriever_llama2_7b_ru_qlora_t4

print("Archive:", ARCHIVE_PATH)

wandb.save(LOG_PATH)
wandb.save(ARCHIVE_PATH)
wandb.finish()
